# 02.09 — Professional Analytics Code

Reconstructed learning notebook · 6 November 2026

**Source:** [Long-Term Analytics Curriculum](https://chatgpt.com/c/6a5e9e66-45d8-83eb-ba05-f22d3010f1ed).

This notebook reconstructs the documented chapter in its original lesson order. The linked conversation was read in the browser after the chat-reading tool returned only its five latest turns. The recovered chapter includes tasks, examples, review explanations, historical attempts, corrected solutions, datasets, knowledge checks, and tests.

**Source boundaries:** This is a reconstruction of the documented learning content, not a byte-for-byte recovery of the deleted notebook. The original cell boundaries, execution counts, and full screenshot-only submissions are not available. Where a screenshot was described by the original assistant, that description is identified as the evidence; unseen screenshot code is not invented. Early-lesson summaries below are editorial summaries of the recovered messages. Repeated feedback is consolidated without introducing new exercises.

**How to use this notebook:** Read the instruction cells, try each task, then compare the historical attempt and corrected code. Run code cells from top to bottom. Historical incomplete or incorrect code is preserved in Markdown so it remains visible during a successful Run All. Selected failures are reproduced inside explicit exception handlers. Fresh execution output is labelled as reconstruction-time output rather than original output.

**Execution support added during reconstruction:** imports; isolated temporary module files; small demonstrations that catch expected failures; source export for pytest; execution checks; and cleanup. These are labelled separately. They make the notebook portable and runnable without claiming to be original course material. No external dataset is required. The notebook uses Python 3.10+ with `pandas`, `pytest`, and an IPython/Jupyter kernel.

The historical estimate was **3–4 hours**, explicitly described in the chat as a proposed teaching breakdown rather than a verified curriculum-file duration. The six lesson estimates were 30, 30, 30–40, 30–40, 30–40, and 45–60 minutes respectively.

| Lesson | Topic | Recovered content |
| --- | --- | --- |
| 02.09.01 | Reusable Functions & Type Hints | Three exercises; validation, operators, docstrings, defaults |
| 02.09.02 | Error Handling & Input Validation | Four exercises; conversion, exceptions, business validation, Pandas apply |
| 02.09.03 | Clean Code & Refactoring | Three exercises; aggregate metrics, regions, dictionary comprehensions |
| 02.09.04 | Modular Code & Project Structure | Four exercises/checks; imports, packages, Jupyter search paths |
| 02.09.05 | Automated Testing with pytest | Four exercises and final knowledge check |
| 02.09.06 | Professional Analytics Coding Challenge | Three exercises; shipment analysis, validation, tests |

## Setup

**Reconstruction execution support.** Select your analytics Python environment in Jupyter or VS Code. If required, install `pandas`, `pytest`, and `ipykernel` into that environment before running the notebook. No cell installs software automatically.

In [1]:
import math
import pandas as pd
import pytest


## 02.09.01 — Reusable Functions & Type Hints

### Early-Lesson Summary

Turn repeated KPI calculations into functions. Type hints document expected inputs and outputs but do not enforce them at runtime. Validate the visitor count and purchase count explicitly, describe the business meaning in a docstring, and use a default argument to choose decimal or percentage output.

### Why Functions Matter

The original example repeated `purchases_A / visitors_A`, `purchases_B / visitors_B`, and `purchases_C / visitors_C`. A function centralizes that logic and can be reused for different experiments.

```python
conversion_A = purchases_A / visitors_A
conversion_B = purchases_B / visitors_B
conversion_C = purchases_C / visitors_C

def conversion_rate(purchases, visitors):
    return purchases / visitors

conversion_A = conversion_rate(120, 1000)
conversion_B = conversion_rate(150, 1000)
```

`def` defines a function; `purchases: int` and `visitors: int` document integer inputs; `-> float` documents the returned value; `return` sends the result to the caller. Type annotations support readability, static analysis, and editor tools.

An unvalidated call with zero visitors raises `ZeroDivisionError`. A call with 150 purchases and 100 visitors returns `1.5`, which violates the assumption that each visitor converts at most once.

In [2]:
def conversion_rate(purchases: int, visitors: int) -> float:
    return purchases / visitors


print(conversion_rate(120, 1000))
print(conversion_rate(150, 1000))
print(conversion_rate(150, 100))


0.12
0.15
1.5


### Expected Failure — Unvalidated Division

**Reconstruction execution support:** reproduce the documented exception without interrupting Run All. The exception text printed below comes from the current Python runtime.

In [3]:
try:
    conversion_rate(120, 0)
except ZeroDivisionError as error:
    print(f"{type(error).__name__}: {error}")


ZeroDivisionError: division by zero


### Exercise 02.09.01.01 — Build a Robust KPI Function

Implement a function that accepts integer `purchases` and `visitors`, returns a float conversion rate, raises `ValueError` when `visitors <= 0`, and rejects `purchases < 0` or `purchases > visitors`. Zero purchases is valid.

The first prompt named the function `calculate_conversion_rate`; the subsequent user attempts and reviews used `conversion_rate`. Both names are preserved in their respective contexts.

```python
def calculate_conversion_rate(purchases: int, visitors: int) -> float:
    # Validate number of visitors
    # Validate number of purchases
    # Calculate and return conversion rate
    ...
```

Expected valid calls: `(120, 1000) -> 0.12`, `(150, 1000) -> 0.15`, and `(0, 100) -> 0.0`. Calls `(150, 100)` and `(10, 0)` must raise `ValueError`.

### Historical User Solution — Incomplete try Block

```python
def conversion_rate(purchases :int, visitors: int) -> float:
    try:
        return purchases / visitors
```

The `try` block is incomplete: Python requires an `except` or `finally` clause. Input validation is also missing. Use `if` to inspect a condition and `raise ValueError(...)` to reject it; use `try/except` to handle an exception that occurs during execution. The original feedback described the syntax problem without supplying a full traceback.

### Historical User Solution — Bitwise OR in a Scalar Condition

```python
def conversion_rate(purchases: int, visitors: int) -> float:

    if visitors <= 0:
        raise ValueError("Visitors must be positive")

    if purchases < 0 | purchases > visitors:
        raise ValueError("Purchases must be greater than 0 or less than visitors")

    return purchases / visitors
```

The scalar condition needs `or`. Bitwise `|`, operator precedence, and comparison chaining change the meaning of the attempted expression. The corrected error message describes the inclusive valid purchase range.

### Corrected Solution — Robust Conversion Rate

For scalar Python conditions use `and`, `or`, and `not`. For element-wise Pandas Series conditions use `&`, `|`, and `~`, and put each comparison in parentheses:

```python
mask = (df["purchases"] < 0) | (df["purchases"] > df["visitors"])
```

In [4]:
def conversion_rate(purchases: int, visitors: int) -> float:
    if visitors <= 0:
        raise ValueError("Visitors must be positive")
    if purchases < 0 or purchases > visitors:
        raise ValueError("Purchases must be between 0 and visitors")
    return purchases / visitors


print(conversion_rate(120, 1000))
print(conversion_rate(150, 1000))
print(conversion_rate(0, 100))


0.12
0.15
0.0


In [5]:
for purchases, visitors in [(150, 100), (10, 0)]:
    try:
        conversion_rate(purchases, visitors)
    except ValueError as error:
        print(f"ValueError: {error}")


ValueError: Purchases must be between 0 and visitors
ValueError: Visitors must be positive


### Exercise 02.09.01.02 — Add a Docstring

Add an English docstring with a brief description and `Args`, `Returns`, and `Raises` sections. Keep the function logic unchanged. Describe what the parameters mean in the business context and the exact conditions that cause an exception.

### Historical User Solution — First English Docstring

```python
def conversion_rate(purchases: int, visitors: int) -> float:
    """
    A function for returning different conversina rates

    Args:
        purchases (int): purchase metrics must be integers
        visitors (int): vistor metrics must be integers

    Returns:
        float: the conversion rate muste me retunred as float

    Raises:
        ValueError: we hraise Value Errors to validate the input
    """

    if visitors <= 0:
        raise ValueError("Visitors must be positive")

    if purchases < 0 or purchases > visitors:
        raise ValueError("Purchases must be between 0 and visitors")

    return purchases / visitors
```

The review marked the exercise passed because all four documentation sections and the correct logic were present. The improvement was precision: explain parameter meaning, return range, and validation rules rather than restating types. Historical spelling is retained here.

In [6]:
def conversion_rate(purchases: int, visitors: int) -> float:
    """
    Calculate the purchase conversion rate.

    Args:
        purchases (int): Number of completed purchases.
        visitors (int): Total number of visitors.

    Returns:
        float: Conversion rate between 0.0 and 1.0.

    Raises:
        ValueError: If visitors is not positive, or purchases
            is negative or exceeds the number of visitors.
    """
    if visitors <= 0:
        raise ValueError("Visitors must be positive")
    if purchases < 0 or purchases > visitors:
        raise ValueError("Purchases must be between 0 and visitors")
    return purchases / visitors


### Exercise 02.09.01.03 — Reusable Functions with Default Arguments

Extend `conversion_rate()` with `as_percentage: bool = False`. Keep validation, calculate the decimal rate, and multiply by 100 only when percentage mode is requested. The introductory default-argument example was:

```python
def greet(name: str, uppercase: bool = False) -> str:
    if uppercase:
        return name.upper()
    return name
```

Expected calls: `conversion_rate(120, 1000) -> 0.12`, `conversion_rate(120, 1000, as_percentage=True) -> 12.0`, and `conversion_rate(150, 1000, as_percentage=True) -> 15.0`.

**Historical screenshot feedback:** the original assistant reported an `IndentationError` because the function header had no indented body and the `print()` calls were unindented. The complete screenshot-only source and exact traceback are unavailable; no unseen lines are supplied as an original submission. A function body must be indented immediately after its header. Editing a cell alone does not execute the edited definition.

### Corrected Final Solution — Default Argument

The following function is the complete working version supplied in the chat after that error. The user subsequently confirmed that it ran successfully.

In [7]:
def conversion_rate(
    purchases: int,
    visitors: int,
    as_percentage: bool = False
) -> float:
    if visitors <= 0:
        raise ValueError("Visitors must be positive")
    if purchases < 0 or purchases > visitors:
        raise ValueError("Purchases must be between 0 and visitors")
    rate = purchases / visitors
    if as_percentage:
        return rate * 100
    return rate


print(conversion_rate(120, 1000))
print(conversion_rate(120, 1000, as_percentage=True))
print(conversion_rate(150, 1000, as_percentage=True))


0.12
12.0
15.0


## 02.09.02 — Error Handling & Input Validation

### Early-Lesson Summary

Catch specific conversion exceptions, return cleaned values rather than printing them, and separate technical conversion from business rules. Extend `parse_revenue()` from strings to mixed inputs, reject negative/non-finite revenue, preserve zero, and pass the function itself to `Series.apply()`.

### Why Error Handling Matters

A shipment pipeline may encounter negative delivery times, missing shipment IDs, invalid revenue, or wrong data types. Handle these cases predictably. Avoid bare `except:` clauses that conceal programming errors.

`try` executes potentially failing code; `except` handles a particular exception; `else` runs only when the `try` succeeds; `finally` runs after the attempt regardless of success or failure. The documented example prints `Cannot divide by zero.` followed by `Processing finished.`

In [8]:
try:
    result = 100 / 0
except ZeroDivisionError:
    print("Cannot divide by zero.")
else:
    print("Calculation successful.")
finally:
    print("Processing finished.")


Cannot divide by zero.
Processing finished.


### Exercise 02.09.02.01 — Robust Revenue Calculation

Create `parse_revenue(value: str) -> float | None`. Convert valid strings with `float(value)`, catch `ValueError`, and return `None` when conversion fails.

The starter docstring was “Convert a revenue string to a float. Return None if conversion fails.” Expected values are `"125.50" -> 125.5`, `"80" -> 80.0`, `"invalid" -> None`, and `"" -> None`.

### Historical User Solution — Returning the Original String

```python
def parse_revenue(value: str) -> float | None:
    """
    Convert a revenue string to a float.

    Return None if conversion fails.
    """

    try:
        float(value)
        return value
    except ValueError:
        print("Algorythm found an error")


print(parse_revenue("125.50"))
print(parse_revenue("80"))
print(parse_revenue("invalid"))
print(parse_revenue(""))
```

`float(value)` creates a float but does not replace `value`. Return the conversion result. `print()` displays information; `return` determines the caller’s result. Printing in the exception handler implicitly returns `None`, but an explicit `return None` communicates the intended behavior. The historical message spelling is retained.

In [9]:
def parse_revenue(value: str) -> float | None:
    """Convert a revenue string to a float; return None if conversion fails."""
    try:
        return float(value)
    except ValueError:
        return None


for value in ["125.50", "80", "invalid", ""]:
    print(parse_revenue(value))


125.5
80.0
None
None


### Exercise 02.09.02.02 — Handling Multiple Exception Types

Use `revenues = ["125.50", "80", "invalid", None, 250]`. `float(None)` raises `TypeError`, whereas invalid numeric strings raise `ValueError`. Catch both in one clause, then apply the function with a list comprehension.

Expected result: `[125.5, 80.0, None, None, 250.0]`.

### Historical User Solution — Mixed Inputs and a Duplicate Conversion

```python
def parse_revenue(value: str) -> float | None:
    """
    Convert a revenue string to a float.

    Return None if conversion fails.
    """

    try:
        float(value)
        return float(value)
    except (ValueError, TypeError):
        return None


revenues = ["125.50", "80", "invalid", None, 250]
cleaned_revenues = [parse_revenue(value) for value in revenues]
print(cleaned_revenues)
```

The review marked this passed. Remove the unused first conversion and update the annotation to include strings, integers, floats, and `None`. The tuple in the `except` clause names the alternative exception types.

In [10]:
def parse_revenue(value: str | int | float | None) -> float | None:
    try:
        return float(value)
    except (ValueError, TypeError):
        return None


revenues = ["125.50", "80", "invalid", None, 250]
cleaned_revenues = [parse_revenue(value) for value in revenues]
print(cleaned_revenues)


[125.5, 80.0, None, None, 250.0]


### Exercise 02.09.02.03 — Data Validation vs. Error Handling

`float("-250")` succeeds technically, but negative values violate this sales-revenue dataset’s business rule. Convert numeric inputs, return `None` for invalid/missing values or negative revenue, and return the converted value otherwise. **Zero revenue is valid in this exercise.**

Use `["125.50", "-50", "invalid", None, 250, 0]`. Expected output: `[125.5, None, None, None, 250.0, 0.0]`.

### Historical User Solution — Negative-Revenue Validation

```python
def parse_revenue(value: str | int | float | None) -> float | None:
    try:
        revenue = float(value)

        if revenue < 0:
            return None
        else:
            return revenue

    except (ValueError, TypeError):
        return None


revenues = ["125.50", "-50", "invalid", None, 250, 0]
cleaned_revenues = [parse_revenue(value) for value in revenues]
print(cleaned_revenues)
```

The review marked this passed. An `else` after an unconditional `return` is unnecessary because the return exits the function. The review also noted that `float("nan")` and `float("inf")` do not raise conversion exceptions; `math.isfinite()` can reject those values.

In [11]:
def parse_revenue(value: str | int | float | None) -> float | None:
    try:
        revenue = float(value)
        if revenue < 0:
            return None
        return revenue
    except (ValueError, TypeError):
        return None


revenues = ["125.50", "-50", "invalid", None, 250, 0]
print([parse_revenue(value) for value in revenues])


[125.5, None, None, None, 250.0, 0.0]


### Exercise 02.09.02.04 — Validate a DataFrame

Use the shipment dataset below and create `clean_revenue` by applying `parse_revenue()` to each value of the `revenue` Series. Pandas generally represents missing values in a numeric float column as `NaN`, even when the function returns `None`.

Expected clean values: `150.5`, `NaN`, `NaN`, `NaN`, `300.0`.

In [12]:
shipments = pd.DataFrame({
    "shipment_id": [101, 102, 103, 104, 105],
    "revenue": ["150.50", "-20", "invalid", None, "300"]
})
print(shipments)


   shipment_id  revenue
0          101   150.50
1          102      -20
2          103  invalid
3          104      NaN
4          105      300


### Historical User Solution — Calling the Function Inside DataFrame.apply

```python
shipments["clean_revenue"] = shipments.apply(parse_revenue(shipments["revenue"]))
```

This calls `parse_revenue()` immediately on an entire Series, and then tries to apply its result to the whole DataFrame. Select the revenue Series and pass the function reference: `shipments["revenue"].apply(parse_revenue)`. `parse_revenue(value)` executes now; `parse_revenue` refers to the callable; `.apply(parse_revenue)` calls it separately for every Series value.

### Knowledge Check — Passing a Function to apply

Why write `.apply(parse_revenue)` rather than `.apply(parse_revenue())`?

A. Because `.apply()` only accepts functions without parameters.  
B. Because we pass the function itself, allowing Pandas to call it separately for each value.  
C. Because parentheses are not allowed inside Pandas operations.  
D. Because `.apply()` automatically converts every value into a float.

**Correct answer: B.** A textual user answer to this check is not documented.

### Historical Jupyter State Problem

The next screenshot feedback reported that the `.apply()` syntax was correct but **all** cleaned values were missing, including `"150.50"` and `"300"`. The original assistant suspected an older `parse_revenue()` definition in the kernel and asked for `print(parse_revenue("150.50"))` and `print(parse_revenue("300"))`, whose expected results are `150.5` and `300.0`.

Jupyter retains the last **executed** function definition. Editing a cell does not update the running function until that cell is executed again. The supplied fix combined the complete definition and dataset in one cell. Restart → Run All was also described as a way to rebuild state in order. The user then confirmed that the code was working.

### Corrected Final Solution — Finite, Non-Negative Revenue

This complete version, including the finite-value check, was documented in the chat.

In [13]:
def parse_revenue(value: str | int | float | None) -> float | None:
    try:
        revenue = float(value)
        if not math.isfinite(revenue) or revenue < 0:
            return None
        return revenue
    except (ValueError, TypeError):
        return None


print(parse_revenue("150.50"))
print(parse_revenue("300"))

shipments = pd.DataFrame({
    "shipment_id": [101, 102, 103, 104, 105],
    "revenue": ["150.50", "-20", "invalid", None, "300"]
})
shipments["clean_revenue"] = shipments["revenue"].apply(parse_revenue)
print(shipments)


150.5
300.0
   shipment_id  revenue  clean_revenue
0          101   150.50          150.5
1          102      -20            NaN
2          103  invalid            NaN
3          104      NaN            NaN
4          105      300          300.0


## 02.09.03 — Clean Code & Refactoring

### Early-Lesson Summary

Refactoring improves code structure while preserving its intended behavior. Use descriptive variable names, one logical responsibility per function, reusable business calculations, readable formatting, and fewer hardcoded assumptions. DRY means “Don’t Repeat Yourself.”

The original working-but-unclear example was:

```python
df = pd.DataFrame({
    "shipment_id": [101, 102, 103, 104],
    "revenue": [100, 200, 150, 250],
    "cost": [60, 120, 90, 180]
})
x = df["revenue"].sum()
y = df["cost"].sum()
z = x - y
a = z / x
print(x, y, z, a)
```

### Exercise 02.09.03.01 — Refactor Analytics Code

Rewrite the calculation as `calculate_profit_metrics(df: pd.DataFrame) -> dict`. Use meaningful variable names. Return a dictionary with `total_revenue`, `total_cost`, `total_profit`, and `profit_margin`. Calculate margin as total profit divided by total revenue. Return the metrics instead of printing inside the function.

Expected totals: revenue `700`, cost `450`, profit `250`, margin `0.35714285714285715` (35.71%).

In [14]:
df = pd.DataFrame({
    "shipment_id": [101, 102, 103, 104],
    "revenue": [100, 200, 150, 250],
    "cost": [60, 120, 90, 180]
})


### Historical User Solution — Calculations Without a Return

```python
def calculate_profit_metrics(df: pd.DataFrame) -> dict:
    total_revenue = df["revenue"].sum()
    total_cost = df["cost"].sum()
    total_profit = total_revenue - total_cost
    profit_margin = total_profit / total_revenue
```

All calculations were correct, but the missing `return` makes the function return `None`. The user then added the dictionary below; the review marked all five requirements passed and suggested indenting its entries for readability.

### Historical User Solution — User Solution After Adding the Dictionary

```python
def calculate_profit_metrics(df: pd.DataFrame) -> dict:
    total_revenue = df["revenue"].sum()
    total_cost = df["cost"].sum()
    total_profit = total_revenue - total_cost
    profit_margin = total_profit / total_revenue

    return {
    "total_revenue": total_revenue,
    "total_cost": total_cost,
    "total_profit": total_profit,
    "profit_margin": profit_margin
}
metrics = calculate_profit_metrics(df)

print(metrics)
```

This valid historical formatting is retained here. The repeated dataset/import from the submission is given in the preceding setup cell. The corrected executable version below applies the documented formatting improvement.

In [15]:
def calculate_profit_metrics(df: pd.DataFrame) -> dict:
    total_revenue = df["revenue"].sum()
    total_cost = df["cost"].sum()
    total_profit = total_revenue - total_cost
    profit_margin = total_profit / total_revenue
    return {
        "total_revenue": total_revenue,
        "total_cost": total_cost,
        "total_profit": total_profit,
        "profit_margin": profit_margin
    }


metrics = calculate_profit_metrics(df)
print(metrics)
print(metrics["profit_margin"])


{'total_revenue': np.int64(700), 'total_cost': np.int64(450), 'total_profit': np.int64(250), 'profit_margin': np.float64(0.35714285714285715)}
0.35714285714285715


**Documented limitation:** the review noted that zero aggregate revenue needs an explicit division-by-zero policy and deferred the issue. No final policy for `calculate_profit_metrics()` was documented, so this reconstruction preserves the positive-revenue example rather than inventing an original guard.

### Exercise 02.09.03.02 — Eliminating Repeated Code (DRY Principle)

Use the existing function with the three regional DataFrames below. Store their metrics in `regional_metrics`, with keys `"North"`, `"South"`, and `"East"`. Reuse the function rather than rewriting the profit calculations.

The repeated version illustrated the problem:

```python
north_profit = north["revenue"].sum() - north["cost"].sum()
south_profit = south["revenue"].sum() - south["cost"].sum()
east_profit = east["revenue"].sum() - east["cost"].sum()
```

In [16]:
north = pd.DataFrame({
    "revenue": [100, 200, 300],
    "cost": [60, 120, 180]
})
south = pd.DataFrame({
    "revenue": [150, 250, 350],
    "cost": [100, 180, 240]
})
east = pd.DataFrame({
    "revenue": [200, 400, 600],
    "cost": [150, 250, 350]
})


### Historical User Solution — Regional Metrics (Passed)

The following was the user’s successful solution; its calculations also form the final answer.

In [17]:
regional_metrics = {
    "North": calculate_profit_metrics(north),
    "South": calculate_profit_metrics(south),
    "East": calculate_profit_metrics(east)
}
print(regional_metrics)


{'North': {'total_revenue': np.int64(600), 'total_cost': np.int64(360), 'total_profit': np.int64(240), 'profit_margin': np.float64(0.4)}, 'South': {'total_revenue': np.int64(750), 'total_cost': np.int64(520), 'total_profit': np.int64(230), 'profit_margin': np.float64(0.30666666666666664)}, 'East': {'total_revenue': np.int64(1200), 'total_cost': np.int64(750), 'total_profit': np.int64(450), 'profit_margin': np.float64(0.375)}}


| Region | Revenue | Cost | Profit | Profit margin |
| --- | ---: | ---: | ---: | ---: |
| North | 600 | 360 | 240 | 40.00% |
| South | 750 | 520 | 230 | 30.67% |
| East | 1200 | 750 | 450 | 37.50% |

### Exercise 02.09.03.03 — Refactoring with Dictionary Comprehensions

If there were 100 regions, manually listing 100 calls would be cumbersome. The introductory comprehension example was `squared = {number: number**2 for number in [1, 2, 3]}`, producing `{1: 1, 2: 4, 3: 9}`. General syntax: `{key: value for item in iterable}`.

First create `regions = {"North": north, "South": south, "East": east}`. Replace the manual dictionary with one comprehension. Iterate over `regions.items()`, use the region name as the output key, and pass the corresponding DataFrame to `calculate_profit_metrics()`.

### Historical User Solution — Dictionary Comprehension Reported in Screenshot Feedback

```python
regional_metrics = {
    regions: calculate_profit_metrics(regions) for region in regions
}
```

The original assistant transcribed this expression from a screenshot and reported `KeyError: 'revenue'`. It passes the whole `regions` dictionary where the function expects a DataFrame; its key expression also uses the dictionary rather than a region name. This is the documented review’s error report, not a newly fabricated traceback. Iterating over a dictionary yields keys; `.items()` yields key–value pairs.

In [18]:
regions = {"North": north, "South": south, "East": east}
regional_metrics = {
    region_name: calculate_profit_metrics(region_df)
    for region_name, region_df in regions.items()
}
print(regional_metrics)


{'North': {'total_revenue': np.int64(600), 'total_cost': np.int64(360), 'total_profit': np.int64(240), 'profit_margin': np.float64(0.4)}, 'South': {'total_revenue': np.int64(750), 'total_cost': np.int64(520), 'total_profit': np.int64(230), 'profit_margin': np.float64(0.30666666666666664)}, 'East': {'total_revenue': np.int64(1200), 'total_cost': np.int64(750), 'total_profit': np.int64(450), 'profit_margin': np.float64(0.375)}}


`region_name` is `"North"`, `"South"`, or `"East"`; `region_df` is its corresponding DataFrame. The result is equivalent to the previous three explicit calls.

**Revision request retained from the conversation:** the user asked to proceed while remembering questions about dictionary comprehensions and `.items()` for a later stage. Revisit keys versus key–value pairs and passing DataFrames into functions. The later chapter wrap-up also retained exception-testing logic as a revision topic.

## 02.09.04 — Modular Code & Project Structure

### From Notebooks to Reusable Python Modules

Notebooks support exploration; reusable functions can be placed in `.py` modules and imported by analysis code. The original suggested project contained data, notebooks, modules, tests, and dependency declarations:

```text
logistics_analytics/
├── data/
├── notebooks/
│   └── analysis.ipynb
├── src/
│   ├── metrics.py
│   └── cleaning.py
├── tests/
│   └── test_metrics.py
└── requirements.txt
```

For example, `metrics.py` can define `calculate_profit(revenue: float, cost: float) -> float` with `return revenue - cost`. Another file can use `from metrics import calculate_profit`, then `calculate_profit(1000, 650)`, yielding `350`. Python must be able to locate the module on its import path.

### Exercise 02.09.04.01 — Understanding Imports

Suppose `analysis.py` and `metrics.py` are in the same project directory. `metrics.py` defines `conversion_rate(purchases: int, visitors: int) -> float`, returning `purchases / visitors`. Which statement imports the function into `analysis.py`?

A. `import conversion_rate from metrics`  
B. `from metrics import conversion_rate`  
C. `from analysis import metrics`  
D. `metrics.conversion_rate import`

**Historical user answer: B. Correct answer: B.** Importing the function allows `conversion_rate(120, 1000)`; importing the entire module with `import metrics` allows `metrics.conversion_rate(120, 1000)`. Both styles are valid; the second identifies the providing module explicitly.

### Exercise 02.09.04.02 — Organizing a Python Project

```text
logistics_analytics/
├── analysis.py
└── src/
    ├── __init__.py
    ├── metrics.py
    └── cleaning.py
```

`src/metrics.py` defines `calculate_profit()` as above. `src/cleaning.py` defines:

```python
def clean_customer_name(name: str) -> str:
    return name.strip().title()
```

Import both functions into root-level `analysis.py`. Calculate profit for `(1000, 650)` and clean `"  amazon  "`. Expected output: `350` and `Amazon`.

### Historical User Solution — File Paths Used as Import Syntax

```python
# Import calculate_profit from src.metrics
from src/metrics.py import calculate_profit

# Import clean_customer_name from src.cleaning
from src/cleaning.py import clean_customer_name

# Test the imported functions
profit = calculate_profit(1000, 650)
customer = clean_customer_name("  amazon  ")

print(profit)    # 350
print(customer)  # Amazon
```

The same incorrect import forms appeared in the supplied exercise prompt as well as the user submission. Standard Python imports use dots to separate package/module names and omit the `.py` suffix. Correct forms are `from src.metrics import calculate_profit` and `from src.cleaning import clean_customer_name`. A filesystem path such as `src/utils/helpers.py` corresponds to the module path `src.utils.helpers`.

### Executable Module Example

**Reconstruction execution support:** create the documented example package in a temporary directory. Run imports in a fresh Python process, so a pre-existing `src` package in your notebook does not interfere. These files are examples, not modifications to your project’s actual modules.

In [19]:
import inspect
import os
from pathlib import Path
import subprocess
import sys
import tempfile


module_demo_directory = tempfile.TemporaryDirectory(prefix="chapter_02_09_modules_")
module_demo_root = Path(module_demo_directory.name)
module_demo_src = module_demo_root / "src"
module_demo_src.mkdir()
(module_demo_src / "__init__.py").write_text("", encoding="utf-8")
(module_demo_src / "metrics.py").write_text(
    "def calculate_profit(revenue: float, cost: float) -> float:\n"
    "    return revenue - cost\n\n"
    "def conversion_rate(purchases: int, visitors: int) -> float:\n"
    "    return purchases / visitors\n",
    encoding="utf-8"
)
(module_demo_src / "cleaning.py").write_text(
    "def clean_customer_name(name: str) -> str:\n"
    "    return name.strip().title()\n",
    encoding="utf-8"
)

module_example = """from src.metrics import calculate_profit
from src.cleaning import clean_customer_name

profit = calculate_profit(1000, 650)
customer = clean_customer_name("  amazon  ")
print(profit)
print(customer)
"""
module_result = subprocess.run(
    [sys.executable, "-c", module_example],
    cwd=module_demo_root, text=True, capture_output=True, check=True
)
print(module_result.stdout, end="")


350
Amazon


### Exercise 02.09.04.03 — What Is __init__.py?

What is the primary purpose of `__init__.py` in a package?

A. It automatically executes every Python file inside the directory.  
B. It identifies the directory as a regular Python package and can contain package initialization code.  
C. It installs the package’s dependencies.  
D. It converts Python files into Jupyter notebooks.

**Historical user answer: B. Correct answer: B.** The file can be empty or contain initialization code executed when the package is imported. Modern Python also supports namespace packages without `__init__.py`; explicitly including it is often a useful convention.

### Exercise 02.09.04.04 — Understanding Import Errors in Jupyter

Consider:

```text
data-science-prep/
├── src/
│   ├── __init__.py
│   └── metrics.py
└── notebooks/
    └── analysis.ipynb
```

Running `from src.metrics import calculate_profit` in the notebook may produce:

```text
ModuleNotFoundError: No module named 'src'
```

Python searches directories listed in `sys.path`. If the working directory is `notebooks/`, the parent project directory may not be on that path. Check the current directory and search paths. A long-term solution is an installable package; adding the project root to `sys.path` can help in a quick notebook experiment.

In [20]:
print("Working directory:", Path.cwd())
print("Python search paths:", sys.path)


Working directory: /Users/felixpless/Documents/ChatGPT/Data Science & AI/data-science-prep/02-data-toolkit/09-professional-analytics-code
Python search paths: ['/opt/homebrew/Cellar/python@3.14/3.14.0_1/Frameworks/Python.framework/Versions/3.14/lib/python314.zip', '/opt/homebrew/Cellar/python@3.14/3.14.0_1/Frameworks/Python.framework/Versions/3.14/lib/python3.14', '/opt/homebrew/Cellar/python@3.14/3.14.0_1/Frameworks/Python.framework/Versions/3.14/lib/python3.14/lib-dynload', '', '/Users/felixpless/Library/CloudStorage/OneDrive-Persönlich/Projects/data-science-prep/.venv/lib/python3.14/site-packages']


Your notebook cannot import `src.metrics`, although `metrics.py` exists. What should you investigate first?

A. Whether Pandas is installed.  
B. Whether the project root is available on Python’s module search path (`sys.path`).  
C. Whether the notebook contains at least 100 lines of code.  
D. Whether `metrics.py` has been converted to CSV.

**Historical user answer: B. Correct answer: B.** This completed lesson 4 of 6 in the chat.

## 02.09.05 — Automated Testing with pytest

### Why Automated Tests Matter

Changing a function can accidentally break a revenue, margin, or conversion calculation. Tests check its expected behavior for valid inputs and its intended failures for invalid inputs.

`assert` lets execution continue when its condition is true and raises `AssertionError` when false. `pytest` discovers and runs tests from files such as `test_metrics.py`:

```python
from src.metrics import calculate_profit

def test_calculate_profit():
    assert calculate_profit(1000, 650) == 350
```

From a project root with pytest installed, run `python -m pytest`. Ordinary pytest discovery does **not** collect tests from `.ipynb` files. A notebook can call a plain test function directly; parameterized tests need pytest to provide their inputs. The final notebook cells export the recovered functions/tests to temporary `.py` files and actually run pytest.

In [21]:
def calculate_profit(revenue: float, cost: float) -> float:
    return revenue - cost


assert calculate_profit(1000, 650) == 350
assert calculate_profit(500, 500) == 0
assert calculate_profit(100, 150) == -50
print("Profit assertions passed.")


Profit assertions passed.


### Exercise 02.09.05.01 — Write Your First Unit Tests

Use `conversion_rate(purchases, visitors, as_percentage=False)`. The function shown again in this lesson used `ValueError("Visitors must be positive")` and `ValueError("Invalid number of purchases")`; this differs in wording from lesson 1’s purchase validation message but has the same condition.

Write three assertions: `(120, 1000) -> 0.12`, `(0, 100) -> 0.0`, and `(150, 1000, as_percentage=True) -> 15.0`. Put them in `test_conversion_rate()`, call the function, and print `All tests passed!`.

**Historical evidence:** the user submitted a screenshot, and the original review reported that all three assertions passed. It also mentioned terminal output `4 passed in 0.02s` from the user’s existing test files. Those four tests were explicitly distinguished from the new notebook assertions. The unseen screenshot’s exact source is not reconstructed as an original user submission. The solution below implements the documented assertions.

In [22]:
def conversion_rate(
    purchases: int,
    visitors: int,
    as_percentage: bool = False
) -> float:
    if visitors <= 0:
        raise ValueError("Visitors must be positive")
    if purchases < 0 or purchases > visitors:
        raise ValueError("Invalid number of purchases")
    rate = purchases / visitors
    if as_percentage:
        return rate * 100
    return rate


def test_conversion_rate():
    assert conversion_rate(120, 1000) == 0.12
    assert conversion_rate(0, 100) == 0.0
    assert conversion_rate(150, 1000, as_percentage=True) == 15.0


test_conversion_rate()
print("All tests passed!")


All tests passed!


### Exercise 02.09.05.02 — Testing Exceptions with pytest

Test three invalid input combinations: `(10, 0)`, `(-5, 100)`, and `(150, 100)`. Each must raise `ValueError`. A `with pytest.raises(ValueError):` block passes only if the call inside raises that expected exception. Correct failure behavior is as important as correct success behavior.

Complete three context-manager blocks inside `test_invalid_conversion_rates()`, call the function, and print `All exception tests passed!`.

### Historical User Solution — All Three Exception Checks Passed

The following preserves the user’s test logic; whitespace is normalized and the redundant inline labels are expressed by the exercise instructions.

In [23]:
def test_invalid_conversion_rates():
    with pytest.raises(ValueError):
        conversion_rate(10, 0)
    with pytest.raises(ValueError):
        conversion_rate(-5, 100)
    with pytest.raises(ValueError):
        conversion_rate(150, 100)


test_invalid_conversion_rates()
print("All exception tests passed!")


All exception tests passed!


### Exercise 02.09.05.03 — Parameterized Testing

Refactor the three invalid conversion-rate scenarios into one parameterized test. Use `@pytest.mark.parametrize("purchases, visitors", [...])` and `pytest.raises(ValueError)`.

Parameterization tells pytest to execute the same test separately for each input combination. The introductory profit example used `(1000, 650, 350)`, `(500, 500, 0)`, and `(100, 150, -50)`:

In [24]:
@pytest.mark.parametrize(
    "revenue, cost, expected",
    [(1000, 650, 350), (500, 500, 0), (100, 150, -50)]
)
def test_calculate_profit(revenue, cost, expected):
    assert calculate_profit(revenue, cost) == expected


### Historical User Solution — Parameterized Test Without Assertions

```python
import pytest

@pytest.mark.parametrize(
    "purchases, visitors",
    [
        (10,0),
        (-5,100),
        (150,100)
    ]
)
def test_invalid_conversion_rates(purchases, visitors):

    # Verify that each combination raises ValueError
    ...
```

The decorator and all three input pairs were correct, but the test logic was missing. `...` is valid Python (the `Ellipsis` object); a test containing only that expression can pass without verifying anything. Add the exception context manager and call the function. Run this through pytest rather than calling it with no arguments in a notebook.

In [25]:
@pytest.mark.parametrize(
    "purchases, visitors",
    [(10, 0), (-5, 100), (150, 100)]
)
def test_invalid_conversion_rates(purchases, visitors):
    with pytest.raises(ValueError):
        conversion_rate(purchases, visitors)


### Exercise 02.09.05.04 — Final Testing Challenge

Given `calculate_profit_margin(revenue: float, cost: float) -> float`, reject `revenue <= 0` with `ValueError("Revenue must be positive")`, otherwise return `(revenue - cost) / revenue`.

Write two parameterized tests:

1. `test_valid_profit_margin`: `(1000, 600, 0.4)`, `(500, 500, 0.0)`, `(200, 250, -0.25)`.
2. `test_invalid_profit_margin`: revenue `0` and `-100`, with cost `100` in both calls, must raise `ValueError`.

Use `pytest.approx(expected)` for floating-point results, because floating-point arithmetic may introduce tiny rounding differences.

### Historical User Solution — Indented Decorator and Missing Test Function

```python
def calculate_profit_margin(revenue: float, cost: float) -> float:

    if revenue <= 0:
        raise ValueError("Revenue must be positive")

    return (revenue - cost) / revenue

    @pytest.mark.parametrize(
    "revenue, cost, expected",
    [
        (1000, 600, 0.4),
        (500, 500, 0.0),
        (200, 250, -0.25)
    ]
)
```

The function and three valid data cases were correct. The decorator was indented inside the calculation function and had no following test definition. Place the decorator at the same top-level indentation as `def test_valid_profit_margin(...)`. The review then supplied that test and asked the user to write the invalid-input test.

### Historical User Solution — One Tuple Instead of Two Invalid Cases

```python
def calculate_profit_margin(revenue: float, cost: float) -> float:

    if revenue <= 0:
        raise ValueError("Revenue must be positive")

    return (revenue - cost) / revenue

@pytest.mark.parametrize(
"revenue, cost, expected",
[
    (1000, 600, 0.4),
    (500, 500, 0.0),
    (200, 250, -0.25)
]
)

def test_valid_profit_margin(revenue, cost, expected):
    assert calculate_profit_margin(revenue, cost) == pytest.approx(expected)


@pytest.mark.parametrize(
    "revenue",
    [
        (0,-100)
    ]
)
def test_invalid_profit_margin(revenue):
    # Your exception test
    assert calculate_profit_margin(revenue, cost) == pytest.approx(expected)
```

The valid-input test was correct. For one parameter, `[(0, -100)]` supplies one tuple-valued case rather than two separate revenues. `cost` and `expected` are not defined in the invalid-input test. The expected behavior is an exception, so use `pytest.raises(ValueError)` rather than a numerical assertion.

### Corrected Final Solution — Five Profit-Margin Cases

Use `[0, -100]` for the invalid revenues. Pytest collects three valid and two invalid cases from the following two test functions.

In [26]:
def calculate_profit_margin(revenue: float, cost: float) -> float:
    if revenue <= 0:
        raise ValueError("Revenue must be positive")
    return (revenue - cost) / revenue


@pytest.mark.parametrize(
    "revenue, cost, expected",
    [(1000, 600, 0.4), (500, 500, 0.0), (200, 250, -0.25)]
)
def test_valid_profit_margin(revenue, cost, expected):
    assert calculate_profit_margin(revenue, cost) == pytest.approx(expected)


@pytest.mark.parametrize("revenue", [0, -100])
def test_invalid_profit_margin(revenue):
    with pytest.raises(ValueError):
        calculate_profit_margin(revenue, 100)


### Final Knowledge Check — 02.09.05

What happens when this test runs?

```python
with pytest.raises(ValueError):
    calculate_profit_margin(1000, 600)
```

A. The test passes because the profit margin is 0.4.  
B. The test fails because no `ValueError` is raised.  
C. The test automatically converts the result to a percentage.  
D. The test returns `None` and passes.

**Historical user answer: A. Correct answer: B.** Valid revenue produces `0.4` without an exception, but the context manager requires an exception. The original explanation showed a message similar to:

```text
Failed: DID NOT RAISE <class 'ValueError'>
```

| Goal | Pattern |
| --- | --- |
| Verify a result | `assert result == expected` |
| Compare floating-point values | `assert result == pytest.approx(expected)` |
| Verify an exception | `with pytest.raises(ValueError): ...` |
| Run the same check with multiple inputs | `@pytest.mark.parametrize(...)` |

### Reproducing the Documented Failure Safely

**Reconstruction execution support:** explicitly catch pytest’s failure outcome so Run All can continue. The printed message is from this execution.

In [27]:
try:
    with pytest.raises(ValueError):
        calculate_profit_margin(1000, 600)
except pytest.fail.Exception as error:
    print(f"Failed: {error}")

assert calculate_profit_margin(1000, 600) == pytest.approx(0.4)
with pytest.raises(ValueError):
    calculate_profit_margin(0, 600)


Failed: DID NOT RAISE ValueError


## 02.09.06 — Professional Analytics Coding Challenge

### Scenario — Logistics Profitability Analysis

You work as a Data Analyst at a logistics company. Build a reusable analytics function that evaluates shipment profitability using the original five-row dataset below.

In [28]:
shipments = pd.DataFrame({
    "shipment_id": [101, 102, 103, 104, 105],
    "revenue": [1000, 1500, 800, 2000, 1200],
    "cost": [700, 1100, 900, 1400, 850]
})
print(shipments)


   shipment_id  revenue  cost
0          101     1000   700
1          102     1500  1100
2          103      800   900
3          104     2000  1400
4          105     1200   850


### Exercise 02.09.06.01 — Build the Analytics Function

Implement `analyze_shipments(df: pd.DataFrame) -> pd.DataFrame` with five requirements:

1. Accept a Pandas DataFrame.
2. Calculate `profit = revenue - cost`.
3. Calculate `profit_margin = profit / revenue`.
4. Return the resulting DataFrame.
5. Preserve the original DataFrame.

Expected output (margin rounded here for display):

| shipment_id | revenue | cost | profit | profit_margin |
| ---: | ---: | ---: | ---: | ---: |
| 101 | 1000 | 700 | 300 | 0.3000 |
| 102 | 1500 | 1100 | 400 | 0.2667 |
| 103 | 800 | 900 | -100 | -0.1250 |
| 104 | 2000 | 1400 | 600 | 0.3000 |
| 105 | 1200 | 850 | 350 | 0.2917 |

### Historical User Solution — Passed All Five Requirements

The user’s implementation below was accepted without a logic correction. `df.copy()` avoids mutating the caller’s DataFrame; column arithmetic uses vectorized Pandas operations.

In [29]:
def analyze_shipments(df: pd.DataFrame) -> pd.DataFrame:
    new_df = df.copy()
    new_df["profit"] = new_df["revenue"] - new_df["cost"]
    new_df["profit_margin"] = new_df["profit"] / new_df["revenue"]
    return new_df


result = analyze_shipments(shipments)
print(result)


   shipment_id  revenue  cost  profit  profit_margin
0          101     1000   700     300       0.300000
1          102     1500  1100     400       0.266667
2          103      800   900    -100      -0.125000
3          104     2000  1400     600       0.300000
4          105     1200   850     350       0.291667


### Exercise 02.09.06.02 — Input Validation

Extend the function so that it checks that `revenue` and `cost` exist, raises a meaningful `ValueError` when either is missing, and rejects any `revenue <= 0` with `ValueError`. Preserve the input and the calculations for valid data.

Accessing a missing column directly raises `KeyError`; explicit validation should happen first. The original prompt introduced:

```python
required_columns = {"revenue", "cost"}
if not required_columns.issubset(df.columns):
    raise ValueError("Missing required columns")

if (df["revenue"] <= 0).any():
    raise ValueError("Revenue must be positive")
```

`.any()` returns true when at least one row satisfies the condition. The five starter steps were validate required columns, validate revenue, copy the DataFrame, calculate the two metrics, and return the result.

### Historical User Solution — Revenue and Additional Cost Validation

```python
def analyze_shipments(df: pd.DataFrame) -> pd.DataFrame:
    if (df["revenue"] <= 0).any():
        raise ValueError("Revenue must be positive")
    if (df["cost"] <= 0).any():
        raise ValueError("Cost must be positive")

    new_df = df.copy()
    new_df["profit"] = new_df["revenue"] - new_df["cost"]
    new_df["profit_margin"] = new_df["profit"] / new_df["revenue"]
    return new_df

result = analyze_shipments(shipments)
print(result)
```

The original review scored this partially correct (3/4 requirements): copying, calculations, and revenue validation worked. Required-column validation was missing, so a missing revenue column still raised `KeyError`. The extra strictly-positive cost rule was not required; the review noted that zero costs can be legitimate in some business scenarios. The corrected final solution therefore follows the stated exercise contract.

### Corrected Final Solution — Required Columns Before Value Checks

`required_columns - set(df.columns)` is set difference: it identifies exactly which required columns are absent. With `required_columns = {"revenue", "cost"}` and `actual_columns = {"shipment_id", "revenue"}`, the missing set is `{'cost'}`. Sorting the missing names makes the error message stable.

**Documented scope:** this final function checks column presence and non-positive revenue. Additional policies for input object types, missing/non-numeric values, infinities, or duplicate columns were not documented for this challenge and are not silently added. Negative profit is allowed, as shipment 103 demonstrates. The different zero-revenue rule from lesson 2 is intentional: this function divides by revenue.

In [30]:
def analyze_shipments(df: pd.DataFrame) -> pd.DataFrame:
    required_columns = {"revenue", "cost"}
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")
    if (df["revenue"] <= 0).any():
        raise ValueError("Revenue must be positive")
    new_df = df.copy()
    new_df["profit"] = new_df["revenue"] - new_df["cost"]
    new_df["profit_margin"] = new_df["profit"] / new_df["revenue"]
    return new_df


result = analyze_shipments(shipments)
print(result)

required_columns = {"revenue", "cost"}
actual_columns = {"shipment_id", "revenue"}
print(required_columns - actual_columns)


   shipment_id  revenue  cost  profit  profit_margin
0          101     1000   700     300       0.300000
1          102     1500  1100     400       0.266667
2          103      800   900    -100      -0.125000
3          104     2000  1400     600       0.300000
4          105     1200   850     350       0.291667
{'cost'}


### Exercise 02.09.06.03 — Final Automated Testing Challenge

Write three tests: a valid shipment `(revenue=1000, cost=700)` produces profit `300`; a DataFrame with only `cost` raises `ValueError`; and revenue `0` with cost `700` raises `ValueError`.

The user asked to be guided one test at a time. The conversation then worked through these steps.

#### Step 1 — Test a Valid Shipment

After `result = analyze_shipments(df)`, select the first profit with `result["profit"].iloc[0]`: `result` is the returned DataFrame, `["profit"]` selects its profit Series, and `.iloc[0]` selects the first position. An assertion checks a condition (`assert 10 + 5 == 15` passes; `assert 10 + 5 == 20` fails).

Which assertion checks the intended profit?

A. `assert result["profit"].iloc[0] == 300`  
B. `assert df["revenue"] == 300`  
C. `assert analyze_shipments == 300`  
D. `assert result["cost"].iloc[0] == 300`

**Historical user answer: A. Correct answer: A.** The original assistant then supplied the complete test below.

In [31]:
def test_valid_shipment():
    df = pd.DataFrame({"revenue": [1000], "cost": [700]})
    result = analyze_shipments(df)
    assert result["profit"].iloc[0] == 300


#### Step 2 — Test a Missing Column

Use `pd.DataFrame({"cost": [700]})`. The required failure message shown in the chat was:

```text
ValueError: Missing required columns: ['revenue']
```

Which implementation verifies it?

A. `assert analyze_shipments(df) == ValueError`  
B. `with pytest.raises(ValueError):` followed by an indented `analyze_shipments(df)`  
C. `assert df["revenue"] == 0`  
D. `return ValueError`

**Historical user answer: B. Correct answer: B.** The context manager catches the expected exception. The test passes because the failure behavior matches the contract.

In [32]:
def test_missing_column():
    df = pd.DataFrame({"cost": [700]})
    with pytest.raises(ValueError):
        analyze_shipments(df)


#### Step 3 — Test Invalid Revenue

Use a DataFrame with revenue `0` and cost `700`. The function’s `(df["revenue"] <= 0).any()` guard must raise `ValueError("Revenue must be positive")`. Complete the two exception-test lines.

### Historical User Solution — Invalid Revenue (Passed)

The user supplied this complete final test. The review marked the three testing steps completed, while explicitly stating that the tests still needed to be run together in pytest for final verification.

In [33]:
def test_invalid_revenue():
    df = pd.DataFrame({
        "revenue": [0],
        "cost": [700]
    })
    with pytest.raises(ValueError):
        analyze_shipments(df)


### Chapter Completion and Notebook Testing

The chat marked all six lessons completed after the three test implementations were reviewed. The user then asked whether tests inside a Jupyter notebook would run and wanted to start 02.10 in a fresh notebook. Defining tests in a notebook is valid; ordinary pytest file discovery does not find those cells automatically. The executable export below resolves that practical issue for this reconstruction.

The documented next chapter was **02.10 — Final Analytics Project**, beginning with **02.10.01 — Data Ingestion & Initial Exploration**. Its proposed Logistics Performance Analytics pipeline would import, clean, and validate shipment data; compute KPIs; analyze customers/regions; visualize results; and test reusable functions. No new 02.10 lesson content is invented here.

## Reconstruction Verification — Execute the Recovered Tests

**Added execution support, not an original exercise.** The next cell checks the reconstructed examples against their documented outcomes, including the unchanged input DataFrame. It also prints the documented final validation messages. These checks do not add undocumented policies to the original functions.

In [34]:
assert parse_revenue("125.50") == 125.5
assert parse_revenue(0) == 0.0
assert all(parse_revenue(value) is None for value in ["invalid", None, "-50", "nan", "inf"])
assert calculate_profit_metrics(df)["total_profit"] == 250
assert regional_metrics["North"]["total_profit"] == 240
assert regional_metrics["South"]["total_profit"] == 230
assert regional_metrics["East"]["total_profit"] == 450

shipments_before = shipments.copy(deep=True)
verified_shipments = analyze_shipments(shipments)
pd.testing.assert_frame_equal(shipments, shipments_before)
assert verified_shipments["profit"].tolist() == [300, 400, -100, 600, 350]
assert verified_shipments["profit_margin"].tolist() == pytest.approx(
    [0.3, 400 / 1500, -0.125, 0.3, 350 / 1200]
)

for invalid_shipments in [
    pd.DataFrame({"cost": [700]}),
    pd.DataFrame({"revenue": [0], "cost": [700]})
]:
    try:
        analyze_shipments(invalid_shipments)
    except ValueError as error:
        print(f"ValueError: {error}")
print("Reconstruction example checks passed.")


ValueError: Missing required columns: ['revenue']
ValueError: Revenue must be positive
Reconstruction example checks passed.


### Run pytest from This Notebook

**Added execution support.** Export the current, corrected function definitions and recovered tests to an isolated temporary directory using `inspect.getsource()`, which works for definitions executed in IPython/Jupyter. Run pytest with the **same Python interpreter** as the notebook. No project files are overwritten, no network access is required, and every test outcome is checked.

Expected pytest collection is **15 cases**: three profit examples, one conversion test containing three assertions, three invalid-conversion parameter cases, five margin cases, and the three shipment tests. The earlier unparameterized exception test was also executed directly in its own cell before being replaced by the parameterized version. The deliberate failure demonstration is excluded from this passing suite.

In [35]:
recovered_functions = [
    conversion_rate, calculate_profit, calculate_profit_margin, analyze_shipments
]
recovered_tests = [
    test_calculate_profit,
    test_conversion_rate,
    test_invalid_conversion_rates,
    test_valid_profit_margin,
    test_invalid_profit_margin,
    test_valid_shipment,
    test_missing_column,
    test_invalid_revenue
]

with tempfile.TemporaryDirectory(prefix="chapter_02_09_pytest_") as test_directory:
    test_root = Path(test_directory)
    module_source = "import pandas as pd\n\n" + "\n\n".join(
        inspect.getsource(function) for function in recovered_functions
    )
    test_source = (
        "import pandas as pd\nimport pytest\n"
        "from recovered_analytics import conversion_rate, calculate_profit, "
        "calculate_profit_margin, analyze_shipments\n\n"
        + "\n\n".join(inspect.getsource(test) for test in recovered_tests)
    )
    (test_root / "recovered_analytics.py").write_text(module_source, encoding="utf-8")
    (test_root / "test_recovered_analytics.py").write_text(test_source, encoding="utf-8")
    test_environment = os.environ.copy()
    test_environment["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"
    pytest_result = subprocess.run(
        [sys.executable, "-m", "pytest", "-v", "--color=no", "-p", "no:cacheprovider",
         "test_recovered_analytics.py"],
        cwd=test_root, env=test_environment,
        capture_output=True, text=True, timeout=120
    )
    print(pytest_result.stdout, end="")
    if pytest_result.stderr:
        print(pytest_result.stderr, end="")
    if pytest_result.returncode != 0:
        raise RuntimeError(f"Recovered pytest suite failed: exit {pytest_result.returncode}")


============================= test session starts ==============================
platform darwin -- Python 3.14.0, pytest-9.1.1, pluggy-1.6.0 -- /Users/felixpless/Library/CloudStorage/OneDrive-Persönlich/Projects/data-science-prep/.venv/bin/python
rootdir: /private/var/folders/qh/s8jt1blj67qfbyh8c_bwr6yc0000gn/T/chapter_02_09_pytest_6oo5r5eh
collecting ... collected 15 items

test_recovered_analytics.py::test_calculate_profit[1000-650-350] PASSED  [  6%]
test_recovered_analytics.py::test_calculate_profit[500-500-0] PASSED     [ 13%]
test_recovered_analytics.py::test_calculate_profit[100-150--50] PASSED   [ 20%]
test_recovered_analytics.py::test_conversion_rate PASSED                 [ 26%]
test_recovered_analytics.py::test_invalid_conversion_rates[10-0] PASSED  [ 33%]
test_recovered_analytics.py::test_invalid_conversion_rates[-5-100] PASSED [ 40%]
test_recovered_analytics.py::test_invalid_conversion_rates[150-100] PASSED [ 46%]
test_recovered_analytics.py::test_valid_profit_margin[100

In [36]:
module_demo_directory.cleanup()
print("Temporary module examples cleaned up.")


Temporary module examples cleaned up.


## Source Notes and Remaining Historical Gaps

- The six-lesson order, exercise numbers, all numeric example datasets, known failure messages, and recovered historical attempts come from the referenced conversation. Markdown wording and spacing are cleaned for notebook reading; executable snippets may normalize whitespace and omit inline teaching labels that are already explained in Markdown.
- The chat-reading connector initially exposed only the final five turns about lost cells. The chapter itself was recovered by paging through the original conversation in the browser, including reloading initially unavailable assistant responses.
- Screenshot-only submissions in lessons 1, 2, 3, and 5 are represented by their documented review/transcription. A complete original screenshot, deleted-notebook cell sequence, and exact hidden traceback are not invented. Specific historical outputs such as the prior `4 passed in 0.02s` are quotes from the earlier review, not results of this notebook.
- The early-lesson summaries, source notes, temporary-file support, and verification cells were written for this reconstruction. They are not claimed to be verbatim original course text.
- The zero-total-revenue policy for `calculate_profit_metrics()` was deferred in the original discussion. The shipment challenge’s final validation is limited to its documented requirements. Those limitations are preserved explicitly.
- Revision topics retained: dictionary comprehensions, `.items()`, keys versus key–value pairs, DataFrames passed to functions, and distinguishing expected results from expected exceptions.
- The conversation described cells lost while moving chapter 02.09 out of `statistics_probability.ipynb`. This reconstruction is delivered as a separate file; that source notebook is never written by any cell here.